### Scikit-learn Tutorial 
This tutorial focuses on key elements of scikit-learn, including its design philosophy and usage principles. 
- Design principles of scikit-learn 
    - Consistency: All objects follow a uniform API: fit, predict, transform etc
    - Modularity: Algorithms are decoupled from data. Pipelines help to combine them 
    - Composition: Can chain transformers and models into workflows using Pipeline 
    - Non-proliferation of classes: Few, general purpose classes instead of many specialised ones 
    - Sensible Defaults: Many models work out-of-the-box with good defaults
- Core Objects/Interfacts (4 of them)
    - Estimator: anything with .fit() -> learns from the data 
        - E.g. LinearRegression, KMeans, PCA
    - Transformer: anything with .transform() -> modifies or projects data 
        - E.g. StandardScaler, TfidVectorizer 
    - Predictor: anything with .predict() -> makes predictions 
        - E.g. RandomForestClassifier, SVR
    - Meta-Estimators: Wrappers that modify other estimators 
        - E.g. GridSearchCV, RandomizedSearchCV, OnevsRestClassifier, BaggingClassifier, VotingClassifier, Pipeline
- Core Libraries and their functions 
    - sklearn.datasets -> Load built-in or external datasets 
    - sklearn.model_selection -> Cross-Validation, train-test split, hyperparameter tuning 
    - sklearn.preprocessing -> Feature scaling, encoding, normalisation 
    - sklearn.pipeline -> Chain multiple steps together 
    - sklearn.linear_model, sklearn.tree, sklearn.svm, sklearn.ensemble -> Machine Learning algorithms 
    - sklearn.metrics -> accuracy, precision, confusion matrix 
    - sklearn.decomposition -> dimensionality reduction (e.g. PCA)
    - sklearn.feature_selection -> select important features 
- Main workflow 
    - Raw Data (X, y) -> Preprocessing (e.g. StdScaler) -> Estimator (fit/predict/transform)
    - Wrap this all up in a pipeline that does 
        - X_temp = X_train 
        - for step in pipeline[:-1]: #all transformers 
            - X_temp = step.fit_transform(X_temp, y_train)
        - pipeline[-1].fit(X_temp, y_train )
- Under the hood, scikit-learn runs almost entirely on NumPy arrays 
    - Inputs like X and y are NumPy arrays, even if pass pandas Dataframes
    - Transformers and models operate on NumPy, not pandas 
    - Step-by-step breakdown of below code 
        - load_breast_cancer(return_X_y=True): returns X: numpy.ndarray, shape = (569,30). y: numpy.ndarray, shape=(569,)
        - train-test-split also returns numpy.ndarray
        - StandardScaler().fit_transform(X_train) is computed using Numpy broadcasting 
        - LogisticRegression().fit(): relies on both SciPy for optimisation and NumPy vector math 
    - Pandas not explicitly needed, but used more for EDA/Inspection/Feature selection can convert back and forth 
        - E.g. df = pd.DataFrame(X, columns=feature_names) / X = df.values
- Note that we use train_test_split first to split into train (for fitting of params and hyperparams tuning) vs testing set (held out only used for final eval). This is used to measure generalisation after model selection
    - If want to do hyperparameter tuning, use GridSearchCV which 
        - Splits X_train internally into k folds, trains on the k-1 fold and validates on the held-out fold, 
        - Rotates until all folds are tested to give a robust estimate of model performance corresponding to a combination of hyperparams
        - Then choose the best hyperparameters based on average CV score acros the folds
    - Then finally evaluate on the test set (true final evaluation), if not will risk overfitting



In [2]:
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, confusion_matrix

#load dataset and split into train vs test 
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25, #betwee 0-1 (proportion). Can also specify absolute values
                                                    train_size=None, #proportion of absolute number of training samples. If only one is given, the other is computed automatically
                                                    random_state=42, #seed for reproducibility
                                                    shuffle=True, #Whether to shuffle before splitting
                                                    stratify = y) #Ensure class distribution is preserved in train/test sets (useful for classification)

#Create pipeline (List of Tuples:) -> one run for each combo of hyperparameters
pipe = Pipeline([ #pipeline first does StandardScaler().fit_transform(X_train), then does LogisticRegression().fit() on the scaled features
    ('scaler', StandardScaler()), #pipeline is a list of tuples: all until the last 1 is a transformer, and the last one if the estimator
    ('selector', SelectKBest(score_func=f_classif)), #Select top k features 
    ('clf', LogisticRegression()) #fit classifier 
])

#Define parameter grid for GridSearchCV 
param_grid = {
    'selector__k': [10, 15, 20, 'all'], 
    'clf__C': [0.01, 0.1, 1.0, 10.0], 
    'clf__penalty': ['l2'], 
    'clf__solver': ['lbfgs']

}

#Wrap pipeline with GridSearchCV
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=-1, scoring='accuracy')
grid.fit(X_train, y_train)

#Evaluate
print("Best Params:", grid.best_params_)
print("Best CV Score", grid.best_score_)

# Final evaluation on test set
y_pred = grid.predict(X_test)
print("\n📊 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n📄 Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, confusion_matrix

#load dataset and split into train vs test 
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.25, #betwee 0-1 (proportion). Can also specify absolute values
                                                    train_size=None, #proportion of absolute number of training samples. If only one is given, the other is computed automatically
                                                    random_state=42, #seed for reproducibility
                                                    shuffle=True, #Whether to shuffle before splitting
                                                    stratify = y) #Ensure class distribution is preserved in train/test sets (useful for classification)

#Create pipeline (List of Tuples:)
pipe = Pipeline([ #pipeline first does StandardScaler().fit_transform(X_train), then does LogisticRegression().fit() on the scaled features
    ('scaler', StandardScaler()), #pipeline is a list of tuples: all until the last 1 is a transformer, and the last one if the estimator
    ('clf', LogisticRegression()) #fit classifier 
])

#Fit and evaluate
pipe.fit(X_train, y_train) #based on MLE (i.e. minimize negative-log-likelihood function)
score = pipe.score(X_test, y_test)
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

#Make inferences given some X_new; pipeline automatically scales it for you
X_new = X_test[0].reshape(1,-1)
pipe.predict(X_new)
pipe.predict_proba(X_new) #logistic regression uses sigmoid function; multi-class will use softmax 
pipe.predict_log_proba(X_new) #this just returns the log probabilities which are more numerically stable 

#Note: the above code is equivalent to the following steps (basic one with just scalar and classifier, excluding the grid search etc)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) #remember to scale both X_train and X_test, but no need to scale y (targets)
X_test_scaled = scaler.transform(X_test) #cannot fit again because we cannot refit the scalar on the test set, which causes data leakage. I.e. cannot give the model access to future information 
clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)
score = clf.score(X_test_scaled, y_test)